# B-RASP programs as tractable circuits

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TuLLI-AI/symprog/blob/main/notebooks/b_rasp.ipynb)

**B-RASP** (Boolean RASP) is a small programming language from [*Masked Hard-Attention Transformers Recognize Exactly the Star-Free Languages*](https://arxiv.org/abs/2310.13897) (Yang, Chiang and Angluin, NeurIPS 2024). Its programs compile into transformers with hard attention, and the paper uses it to show that these transformers recognize exactly the star-free regular languages.

Every value in a B-RASP program is Boolean. So, for strings of a **fixed length** $n$, a program is just a Boolean function of its input symbols, which `symprog` can compile into a tractable circuit (an SDD). The circuit then answers questions about **all** strings of length $n$ at once, without enumerating them:

- How many strings does the program accept? What is the probability that a random string is accepted?
- Which accepted string is closest to a given (noisy) string?
- Do two programs agree on every input? If not, on which strings do they differ?
- Does a property of the program's outputs hold for every input?

This notebook is a **sketch**: B-RASP is embedded in Python (rather than being a `symprog` language of its own), and everything is checked against the paper's examples and brute force.

## Setup

If `symprog` is not installed, the notebook uses the version in this repository.

In [1]:
# %pip install -q symprog
import symprog

import itertools
import time

from symprog import Solver

# B-RASP in a nutshell

The input to a program is a string $w = w_1 \cdots w_n$. All data are **Boolean vectors**, with one value per position $i \in \{1, \dots, n\}$. The initial vectors are $Q_\sigma$ for each symbol $\sigma$, where $Q_\sigma(i)$ is true iff $w_i = \sigma$. A program computes new vectors with two kinds of operations:

- **Position-wise**: $P(i) := R(i)$, a Boolean combination of earlier vectors at position $i$.
- **Attention**: $P(i) := \blacktriangleleft_j [M(i, j), S(i, j)]\; V(i, j) : D(i)$, or the same with $\blacktriangleright_j$. Position $i$ looks for the **leftmost** ($\blacktriangleleft$) or **rightmost** ($\blacktriangleright$) position $j$ that is allowed by the *mask* $M$ (no mask, $j < i$, or $j > i$) and satisfies the *score* $S(i, j)$. If there is one, $P(i)$ is the *value* $V(i, j)$, and otherwise the *default* $D(i)$.

A program **accepts** $w$ if its output vector $Y$ is true at the last position, $Y(n)$.

For a fixed $n$, an attention operation unrolls into an if-then-else chain over the candidate positions, in order:

$$P(i) = \text{if } S(i, j_1) \text{ then } V(i, j_1) \text{ else if } S(i, j_2) \text{ then } V(i, j_2) \text{ else } \cdots \text{ else } D(i).$$

# Embedding B-RASP in Python

A program is written as a Python function of a *builder* `p`, which provides the input vectors `p.Q[σ]` and the two operations, `p.position` and `p.attend`. The same program can then run in two ways:

- `Run(alphabet, word)` runs it on **one string**, with Python Booleans (an interpreter);
- `Compile(alphabet, n)` builds it for **all strings of length $n$**: every vector at every position becomes a variable, defined by a formula (a biconditional) over the input variables `Qσ_i`. Each position holds exactly one symbol, which is the constraint `Xor(Qa_i, Qb_i, ...)` (in `symprog`, `Xor` with several arguments means *exactly one*).

The Boolean helpers `And`, `Or` and `Not` work on both Python Booleans and formula strings, and simplify constants away:

In [2]:
def And(*xs):
    if any(x is False for x in xs):
        return False
    xs = [x for x in xs if x is not True]
    return True if not xs else xs[0] if len(xs) == 1 else f"And({', '.join(xs)})"

def Or(*xs):
    if any(x is True for x in xs):
        return True
    xs = [x for x in xs if x is not False]
    return False if not xs else xs[0] if len(xs) == 1 else f"Or({', '.join(xs)})"

def Not(x):
    return (not x) if isinstance(x, bool) else f"Not({x})"

def ITE(condition, then, otherwise):
    return Or(And(condition, then), And(Not(condition), otherwise))

In [3]:
class Vector:
    """A Boolean vector, indexed by positions 1..n."""
    def __init__(self, values):
        self.values = values

    def __getitem__(self, i):
        return self.values[i - 1]

class Builder:
    """The B-RASP operations. Subclasses decide how a new vector is stored (`define`)."""

    def position(self, name, f):
        """P(i) := f(i), a Boolean combination of vectors at position i."""
        return self.define(name, [f(i) for i in self.positions])

    def attend(self, name, choose, mask, score, value, default):
        """P(i) := choose_j [mask(i, j), score(i, j)] value(i, j) : default(i),
        where `choose` is "leftmost" or "rightmost"."""
        exprs = []
        for i in self.positions:
            candidates = [j for j in self.positions if mask(i, j)]
            if choose == "rightmost":
                candidates = candidates[::-1]
            ## the first candidate whose score holds gives the value, else the default
            expr = default(i)
            for j in reversed(candidates):
                expr = ITE(score(i, j), value(i, j), expr)
            exprs.append(expr)
        return self.define(name, exprs)

class Run(Builder):
    """Runs a program on one string."""
    def __init__(self, alphabet, word):
        self.positions = range(1, len(word) + 1)
        self.Q = {a: Vector([c == a for c in word]) for a in alphabet}
        self.vectors = {f"Q{a}": self.Q[a] for a in alphabet}

    def define(self, name, values):
        self.vectors[name] = Vector(values)
        return self.vectors[name]

class Compile(Builder):
    """Builds a program for all strings of length n: one variable per vector and position,
    defined by a formula over the input variables `Qσ_i`."""
    def __init__(self, alphabet, n, prefix=""):
        self.alphabet, self.n, self.prefix = alphabet, n, prefix
        self.positions = range(1, n + 1)
        self.Q = {a: Vector([f"Q{a}_{i}" for i in self.positions]) for a in alphabet}
        self.inputs = [f"Q{a}_{i}" for i in self.positions for a in alphabet]
        ## each position holds exactly one symbol
        self.formulas = [f"Xor({', '.join(f'Q{a}_{i}' for a in alphabet)})" for i in self.positions]
        self.defined = []

    def define(self, name, exprs):
        names = [f"{self.prefix}{name}_{i}" for i in self.positions]
        for v, e in zip(names, exprs):
            self.formulas.append(v if e is True else f"Not({v})" if e is False else f"Biconditional({v}, {e})")
        self.defined += names
        return Vector(names)

    def word(self, model):
        """The input string of a model (an assignment to the variables)."""
        return "".join(next(a for a in self.alphabet if model[f"Q{a}_{i}"]) for i in self.positions)

Finally, a compiled program becomes a solver. The input variables of each position form a **categorical** group: exactly one is true, and its weight is the probability of that symbol at that position (uniform by default). The defined vectors are **auxiliary** variables, without weights, since they are determined by the input. `constraints` adds formulas, e.g., that the program accepts:

In [4]:
def solver_for(compiled, constraints=(), weights=None):
    """A solver for a compiled program, with extra `constraints`. `weights` gives the
    probability of each input variable (uniform over the alphabet by default)."""
    weights = weights or {v: 1 / len(compiled.alphabet) for v in compiled.inputs}
    solver = Solver.from_kwargs(
        var_list=compiled.inputs + compiled.defined,
        var_weights=weights,
        aux_vars=set(compiled.defined),
        categoricals=set(compiled.inputs),
    )
    solver.add(compiled.formulas + list(constraints))
    return solver

# Example: Dyck-1 of depth 2

The paper's running example is the language $L_{1,2}$ of balanced brackets nested at most two deep, with $\ell$ and $r$ as the left and right brackets (`l` and `r` below). Its program first finds the immediately matched pairs $\ell r$, and then checks that the remaining brackets match:

$$
\begin{aligned}
P_\ell(i) &:= \blacktriangleright_j [j < i, 1]\; Q_\ell(j) : 0 &&\text{(the previous symbol is } \ell \text{)}\\
S_r(i) &:= \blacktriangleleft_j [j > i, 1]\; Q_r(j) : 0 &&\text{(the next symbol is } r \text{)}\\
I(i) &:= (Q_\ell(i) \land S_r(i)) \lor (Q_r(i) \land P_\ell(i)) &&\text{(immediately matched)}\\
B_\ell(i) &:= \blacktriangleright_j [j < i, \neg I(j)]\; Q_\ell(j) : 0\\
A_r(i) &:= \blacktriangleleft_j [j > i, \neg I(j)]\; Q_r(j) : 0\\
C(i) &:= I(i) \lor (Q_\ell(i) \land A_r(i)) \lor (Q_r(i) \land B_\ell(i))\\
Y(i) &:= \blacktriangleright_j [1, \neg C(j)]\; 0 : 1 &&\text{(}C\text{ holds everywhere)}
\end{aligned}
$$

In Python, with the builder:

In [5]:
def dyck_1_2(p):
    Ql, Qr = p.Q["l"], p.Q["r"]
    Pl = p.attend("Pl", "rightmost", lambda i, j: j < i, lambda i, j: True, lambda i, j: Ql[j], lambda i: False)
    Sr = p.attend("Sr", "leftmost", lambda i, j: j > i, lambda i, j: True, lambda i, j: Qr[j], lambda i: False)
    I  = p.position("I", lambda i: Or(And(Ql[i], Sr[i]), And(Qr[i], Pl[i])))
    Bl = p.attend("Bl", "rightmost", lambda i, j: j < i, lambda i, j: Not(I[j]), lambda i, j: Ql[j], lambda i: False)
    Ar = p.attend("Ar", "leftmost", lambda i, j: j > i, lambda i, j: Not(I[j]), lambda i, j: Qr[j], lambda i: False)
    C  = p.position("C", lambda i: Or(I[i], And(Ql[i], Ar[i]), And(Qr[i], Bl[i])))
    Y  = p.attend("Y", "rightmost", lambda i, j: True, lambda i, j: Not(C[j]), lambda i, j: False, lambda i: True)
    return Y

#### Running it on a string

Running the program on the two strings of the paper's Figure 2 gives exactly the vectors shown there (the accepted string `llrrllrlrr`, and the rejected `lrrlllrrrl`):

In [6]:
def show(word, program):
    run = Run("lr", word)
    program(run)
    print(f"{'input':6}", " ".join(word))
    for name, vector in run.vectors.items():
        print(f"{name:6}", " ".join("1" if vector[i] else "0" for i in run.positions))
    return run

## the rows of Figure 2 in the paper
FIGURE_2 = {
    "llrrllrlrr": {"Pl": "0110011010", "Sr": "0110010110", "I": "0110011110", "Bl": "0111011111",
                   "Ar": "1110111110", "C": "1111111111", "Y": "1111111111"},
    "lrrlllrrrl": {"Pl": "0100111000", "Sr": "1100011100", "I": "1100011000", "Bl": "0000111100",
                   "Ar": "1100111100", "C": "1100111100", "Y": "0000000000"},
}
for word, rows in FIGURE_2.items():
    run = show(word, dyck_1_2)
    for name, expected in rows.items():
        assert "".join("1" if run.vectors[name][i] else "0" for i in run.positions) == expected
    print()

input  l l r r l l r l r r
Ql     1 1 0 0 1 1 0 1 0 0
Qr     0 0 1 1 0 0 1 0 1 1
Pl     0 1 1 0 0 1 1 0 1 0
Sr     0 1 1 0 0 1 0 1 1 0
I      0 1 1 0 0 1 1 1 1 0
Bl     0 1 1 1 0 1 1 1 1 1
Ar     1 1 1 0 1 1 1 1 1 0
C      1 1 1 1 1 1 1 1 1 1
Y      1 1 1 1 1 1 1 1 1 1

input  l r r l l l r r r l
Ql     1 0 0 1 1 1 0 0 0 1
Qr     0 1 1 0 0 0 1 1 1 0
Pl     0 1 0 0 1 1 1 0 0 0
Sr     1 1 0 0 0 1 1 1 0 0
I      1 1 0 0 0 1 1 0 0 0
Bl     0 0 0 0 1 1 1 1 0 0
Ar     1 1 0 0 1 1 1 1 0 0
C      1 1 0 0 1 1 1 1 0 0
Y      0 0 0 0 0 0 0 0 0 0



#### Compiling it for all strings of length $n$

For $n = 10$, the program becomes a set of formulas over the 20 input variables, and the solver compiles them into an SDD. Adding the constraint $Y(n)$ restricts it to the accepted strings, so model enumeration lists them:

In [7]:
n = 10
compiled = Compile("lr", n)
Y = dyck_1_2(compiled)
print(f"{len(compiled.formulas)} formulas, {len(compiled.inputs)} input variables, {len(compiled.defined)} defined variables")
print("for example:", compiled.formulas[n + 1])

acceptor = solver_for(compiled, constraints=[Y[n]])
print("\naccepted strings:", sorted(compiled.word(m) for m in acceptor.models()))

80 formulas, 20 input variables, 70 defined variables
for example: Biconditional(Pl_2, Ql_1)

accepted strings: ['llrlrlrlrr', 'llrlrlrrlr', 'llrlrrllrr', 'llrlrrlrlr', 'llrrllrlrr', 'llrrllrrlr', 'llrrlrllrr', 'llrrlrlrlr', 'lrllrlrlrr', 'lrllrlrrlr', 'lrllrrllrr', 'lrllrrlrlr', 'lrlrllrlrr', 'lrlrllrrlr', 'lrlrlrllrr', 'lrlrlrlrlr']


In [8]:
acceptor.PrettyPrinting().dot

'digraph sdd {\noverlap=false;\n689526 [label="+",shape=circle,style=filled,fillcolor=gray95];\nps_689526_0 [label="×",shape=circle,style=filled,fillcolor=gray95];\n689526 -> ps_689526_0 [arrowhead=none];\nps_689526_0 -> 725758 [arrowsize=.50,style=dashed];\nps_689526_0 -> 724831 [arrowsize=.50];\n725758 [label="+",shape=circle,style=filled,fillcolor=gray95];\nps_725758_0 [label="×",shape=circle,style=filled,fillcolor=gray95];\n725758 -> ps_725758_0 [arrowhead=none];\nps_725758_0 -> 711033 [arrowsize=.50,style=dashed];\nps_725758_0 -> 710931 [arrowsize=.50];\n711033 [label="+",shape=circle,style=filled,fillcolor=gray95];\nps_711033_0 [label="×",shape=circle,style=filled,fillcolor=gray95];\n711033 -> ps_711033_0 [arrowhead=none];\nps_711033_0 -> n0_157 [arrowsize=.50,style=dashed];\nps_711033_0 -> n1_95 [arrowsize=.50];\nn0_157 [shape=rectangle,label="C_3"];\nn1_95 [shape=rectangle,label="C_10"];\nps_711033_1 [label="×",shape=circle,style=filled,fillcolor=gray95];\n711033 -> ps_711033_1

To check the compiled programs, we compare them with a direct test of $L_{1,2}$ (a counter that must stay between 0 and 2 and end at 0), on every string of every length up to 10. The model count gives the number of accepted strings, and with uniform symbol probabilities, `Prob` gives the fraction of accepted strings, $\text{count} / 2^n$:

In [10]:
def in_dyck_1_2(word):
    depth = 0
    for c in word:
        depth += 1 if c == "l" else -1
        if not 0 <= depth <= 2:
            return False
    return depth == 0

print(" n  accepted  P(accept)  circuit size  time")
for n in range(1, 30):
    start = time.perf_counter()
    compiled = Compile("lr", n)
    Y = dyck_1_2(compiled)
    acceptor = solver_for(compiled, constraints=[Y[n]])

    accepted = {compiled.word(m) for m in acceptor.models()}
    expected = {"".join(w) for w in itertools.product("lr", repeat=n) if in_dyck_1_2(w)}
    count = int(acceptor.query(mode="Count").inference)
    prob = acceptor.query(mode="Prob").inference.item()
    assert accepted == expected and count == len(expected) and abs(prob - count / 2**n) < 1e-6

    size = acceptor.formula.formula.size()  ## the size of the compiled SDD
    print(f"{n:2d}  {count:8d}  {prob:9.4f}  {size:12d}  {time.perf_counter() - start:.2f}s")

 n  accepted  P(accept)  circuit size  time
 1         0     0.0000             0  0.03s
 2         1     0.2500            44  0.02s
 3         0     0.0000             0  0.02s
 4         2     0.1250           135  0.03s
 5         0     0.0000             0  0.05s
 6         4     0.0625           352  0.07s
 7         0     0.0000             0  0.12s
 8         8     0.0312           646  0.17s
 9         0     0.0000             0  0.20s
10        16     0.0156           965  0.41s
11         0     0.0000             0  0.65s
12        32     0.0078          1163  0.73s
13         0     0.0000             0  0.95s
14        64     0.0039          1963  5.62s
15         0     0.0000             0  4.81s
16       128     0.0020          1787  4.02s
17         0     0.0000             0  29.35s


KeyboardInterrupt: 

Strings of odd length are never balanced, so for odd $n$ the constraint $Y(n)$ is unsatisfiable, and the circuit is just $\bot$ (size 0).

# What the circuit gives us

Everything below works on the compiled circuit for $n = 10$, and never enumerates the $2^{10}$ strings.

#### Conditional probabilities

Given that a random string is accepted, what is the probability of $\ell$ at each position? An accepted string must start with $\ell$ and end with $r$, and each other position turns out to hold $\ell$ in exactly half of the accepted strings:

In [9]:
n = 10
compiled = Compile("lr", n)
Y = dyck_1_2(compiled)
acceptor = solver_for(compiled, constraints=[Y[n]])

for k in range(1, n + 1):
    print(f"P(l at {k:2d} | accepted) = {acceptor.query(query=f'Ql_{k}').inference.item():.3f}")

P(l at  1 | accepted) = 1.000
P(l at  2 | accepted) = 0.500
P(l at  3 | accepted) = 0.500
P(l at  4 | accepted) = 0.500
P(l at  5 | accepted) = 0.500
P(l at  6 | accepted) = 0.500
P(l at  7 | accepted) = 0.500
P(l at  8 | accepted) = 0.500
P(l at  9 | accepted) = 0.500
P(l at 10 | accepted) = 0.000


#### The most likely accepted string

Suppose we observe a string with noise: each symbol is correct with probability 0.8. The observation `lllrrlrrlr` is not in the language (it goes three deep). The most likely accepted string given the observation is the accepted string with the highest weight, i.e., the `ViterbiAssignment` with the observation as weights. With symmetric noise, it is an accepted string at the smallest Hamming distance from the observation:

In [10]:
observed = "lllrrlrrlr"
assert not in_dyck_1_2(observed)

noisy = {f"Q{a}_{i}": 0.8 if observed[i - 1] == a else 0.2 for i in range(1, n + 1) for a in "lr"}
best = solver_for(compiled, constraints=[Y[n]], weights=noisy).query(mode="ViterbiAssignment")
corrected = "".join("l" if best.variables[f"Ql_{i}"] else "r" for i in range(1, n + 1))

distance = lambda u, v: sum(a != b for a, b in zip(u, v))
print("observed: ", observed)
print("corrected:", corrected, f"(distance {distance(observed, corrected)})")

## brute force: the smallest distance to an accepted string
members = ["".join(w) for w in itertools.product("lr", repeat=n) if in_dyck_1_2(w)]
assert in_dyck_1_2(corrected)
assert distance(observed, corrected) == min(distance(observed, w) for w in members)

observed:  lllrrlrrlr
corrected: lrlrllrrlr (distance 2)


#### Do two programs agree?

A small change to a program can be hard to evaluate by testing. Below, the attention in $B_\ell$ looks for the *leftmost* (rather than rightmost) earlier position that is not immediately matched. To compare the two programs on all strings of a given length, we compile both into the same solver (with prefixes for their vectors, and shared inputs) and constrain their outputs to differ, with `Xor`. Any model is a counterexample, and the model count is the number of strings where they disagree:

In [11]:
def dyck_variant(p):
    """dyck_1_2, but with leftmost attention in Bl."""
    Ql, Qr = p.Q["l"], p.Q["r"]
    Pl = p.attend("Pl", "rightmost", lambda i, j: j < i, lambda i, j: True, lambda i, j: Ql[j], lambda i: False)
    Sr = p.attend("Sr", "leftmost", lambda i, j: j > i, lambda i, j: True, lambda i, j: Qr[j], lambda i: False)
    I = p.position("I", lambda i: Or(And(Ql[i], Sr[i]), And(Qr[i], Pl[i])))
    Bl = p.attend("Bl", "leftmost", lambda i, j: j < i, lambda i, j: Not(I[j]), lambda i, j: Ql[j], lambda i: False)
    Ar = p.attend("Ar", "leftmost", lambda i, j: j > i, lambda i, j: Not(I[j]), lambda i, j: Qr[j], lambda i: False)
    C = p.position("C", lambda i: Or(I[i], And(Ql[i], Ar[i]), And(Qr[i], Bl[i])))
    return p.attend("Y", "rightmost", lambda i, j: True, lambda i, j: Not(C[j]), lambda i, j: False, lambda i: True)

for n in [2, 4, 6, 8]:
    both = Compile("lr", n, prefix="A_")
    Y_original = dyck_1_2(both)
    both.prefix = "B_"
    Y_variant = dyck_variant(both)

    disagree = solver_for(both, constraints=[f"Xor({Y_original[n]}, {Y_variant[n]})"])
    examples = sorted(both.word(m) for m in disagree.models())
    print(f"n = {n}: the programs disagree on {int(disagree.query(mode='Count').inference)} strings", examples[:3])

n = 2: the programs disagree on 0 strings []
n = 4: the programs disagree on 0 strings []
n = 6: the programs disagree on 1 strings ['llrrrr']
n = 8: the programs disagree on 6 strings ['llrlrrrr', 'llrrlrrr', 'llrrrlrr']


The programs agree on all strings up to length 4, so testing short strings would miss the difference. Running both programs on a counterexample shows that the variant accepts an unbalanced string:

In [12]:
for program in [dyck_1_2, dyck_variant]:
    run = Run("lr", "llrrrr")
    Y = program(run)
    print(f"{program.__name__:12} accepts llrrrr: {Y[6]}")

dyck_1_2     accepts llrrrr: False
dyck_variant accepts llrrrr: True


# A sequence-to-sequence program: associative recall

B-RASP programs can also compute outputs at every position. The paper's Appendix A gives a program for **associative recall**: the input alternates letters (`a`, `b`, `c`) and numbers (`1`, `2`, `3`). The output copies the letters, and at each number outputs the number that followed the most recent earlier occurrence of the preceding letter, or `?` if there was none. For example, `a3b2b1a2c1a1c3` gives `a?b?b2a3c?a2c1`.

In [13]:
def associative_recall(p):
    Q = p.Q
    ## Px(i): the previous symbol is the letter x
    P = {x: p.attend(f"P{x}", "rightmost", lambda i, j: j < i, lambda i, j: True,
                     lambda i, j, x=x: Q[x][j], lambda i: False) for x in "abc"}
    same_letter = lambda i, j: Or(*[And(P[x][i], P[x][j]) for x in "abc"])
    ## Yd: output the number d, found after the most recent occurrence of the same letter
    outputs = {d: p.attend(f"Y{d}", "rightmost", lambda i, j: j < i, same_letter,
                           lambda i, j, d=d: Q[d][j], lambda i: False) for d in "123"}
    for x in "abc":
        outputs[x] = p.position(f"Y{x}", lambda i, x=x: Q[x][i])
    outputs["?"] = p.position("Yq", lambda i: And(
        Or(*[P[x][i] for x in "abc"]), Not(Or(*[outputs[d][i] for d in "123"]))
    ))
    return outputs

run = Run("abc123", "a3b2b1a2c1a1c3")
outputs = associative_recall(run)
result = "".join(next(symbol for symbol, vector in outputs.items() if vector[i]) for i in run.positions)
print(result)
assert result == "a?b?b2a3c?a2c1"

a?b?b2a3c?a2c1


#### Verifying a property for all inputs

Is the output always well defined, with **exactly one** output symbol at every position? With the circuit, we can check this for every input of a given length: we look for a model of "the input is well formed (letters at odd positions, numbers at even positions) and some position does not have exactly one output". If there is none, the property holds for all well-formed inputs of that length. Without the well-formedness condition, the check finds inputs where it fails:

In [14]:
n = 6
compiled = Compile("abc123", n)
outputs = associative_recall(compiled)

one_output = And(*[f"Xor({', '.join(outputs[s][i] for s in outputs)})" for i in compiled.positions])
well_formed = [Or(*[f"Q{x}_{i}" for x in ("abc" if i % 2 == 1 else "123")]) for i in compiled.positions]

violations = solver_for(compiled, constraints=well_formed + [Not(one_output)])
print("well-formed inputs without exactly one output:", list(violations.models()))

anything = solver_for(compiled, constraints=[Not(one_output)])
example = compiled.word(next(iter(anything.models())))
print("an input that is not well formed, without exactly one output:", example)

well-formed inputs without exactly one output: []
an input that is not well formed, without exactly one output: ab2abb


# Towards a B-RASP language in `symprog`

This sketch embeds B-RASP in Python. A next step would be a `symprog` language, `@Parser(language="brasp")`, like the constraint and probabilistic logic languages. Since the program body must be valid Python syntax, operations could be written as, e.g.:

```python
@Parser(language="brasp")
def dyck():
    Pl = rightmost(j < i, True, Ql[j], default=False)
    I = (Ql[i] & Sr[i]) | (Qr[i] & Pl[i])
    ...
    output(Y)
```

with `prog.execute("llrrllrlrr")` running it on a string (like `Run`), and `prog.inference(n=10, ...)` compiling it for a length (like `Compile`).

**Limitations of this approach:**

- **Fixed lengths.** A circuit covers the strings of one length $n$. Statements about all lengths (e.g., that two programs are equivalent on every input) need other tools: B-RASP programs correspond to formulas of linear temporal logic, and so to finite automata, where such questions are decidable.
- **Size.** Each attention operation unrolls into $O(n^2)$ terms, and the circuit grows with $n$. Above, the Dyck circuit grows moderately up to $n = 10$; the associative recall check takes a few seconds for $n = 6$, but close to a minute for $n = 8$.
- **Booleans only.** RASP programs with numbers (e.g., counting) would need richer encodings.

**Connections** with the rest of `symprog`: weighted inputs turn a program into a probabilistic model over strings, and a compiled acceptor can serve as a constraint in the semantic loss (see `semantic_loss.ipynb`), e.g., to train a model to generate strings in the language.